In [ ]:
# Cell 1 — Mount Drive and install packages
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard_v2'

import os, subprocess
subprocess.run(['pip', 'install',
    'xgboost==2.1.0', 'shap==0.45.0', 'scikit-learn==1.5.0',
    'pandas==2.2.0', 'numpy==1.26.0', 'joblib==1.4.0', '-q'], check=False)

assert os.path.exists(f'{DRIVE_BASE}/features/contract_features.csv'), \
    'contract_features.csv not found — run Notebook 03 first'
assert os.path.exists(f'{DRIVE_BASE}/models/contract_feature_schema.json'), \
    'contract_feature_schema.json not found — run Notebook 03 first'
print('Cell 1 ready.')


In [ ]:
# Cell 2 — Load features and schema, validate inputs
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import xgboost as xgb
from xgboost import XGBClassifier
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MODEL_PATH  = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
SCHEMA_PATH = f'{DRIVE_BASE}/models/contract_feature_schema.json'
EVAL_DIR    = f'{DRIVE_BASE}/evaluation'

with open(SCHEMA_PATH) as f:
    FEATURE_COLS = json.load(f)

df = pd.read_csv(f'{DRIVE_BASE}/features/contract_features.csv')

TOTAL  = len(df)
PHISH  = (df['label'] == 1).sum()
BENIGN = (df['label'] == 0).sum()

assert df.shape[1] == 23,              f'Wrong column count: {df.shape[1]}'
assert PHISH == BENIGN,                f'Imbalanced: phishing={PHISH} benign={BENIGN}'
assert TOTAL >= 1000,                  f'Too few rows: {TOTAL}'
assert df.isnull().sum().sum() == 0,   'Nulls found'
assert len(FEATURE_COLS) == 21,        f'Schema length wrong: {len(FEATURE_COLS)}'
assert all(c in df.columns for c in FEATURE_COLS), 'Missing feature columns'

X = df[FEATURE_COLS].values
y = df['label'].values

assert X.shape[1] == len(FEATURE_COLS), \
    f'Feature count mismatch: {X.shape[1]} vs {len(FEATURE_COLS)}'

print(f'Loaded: {df.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Rows: {TOTAL} | Phishing: {PHISH} | Benign: {BENIGN}')
print(f'Training features:')
for f in FEATURE_COLS:
    print(f'  {f}')
print('Cell 2 validation passed.')


In [ ]:
# Cell 3 — Train/test split
all_idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.30, stratify=y, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Train: {X_train.shape} | phishing: {y_train.sum()} | benign: {(y_train==0).sum()}')
print(f'Test:  {X_test.shape}  | phishing: {y_test.sum()}  | benign: {(y_test==0).sum()}')
assert len(test_idx)  == 600,  f'Test size wrong: {len(test_idx)}'
assert len(train_idx) == 1400, f'Train size wrong: {len(train_idx)}'

os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
np.save(f'{DRIVE_BASE}/models/contract_test_indices.npy', test_idx)
print('Test indices saved.')


In [ ]:
# Cell 4 — XGBoost training with 5-fold cross-validation
scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    tree_method='hist',
    device='cpu',
    random_state=42
)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train,
                         cv=cv, scoring='average_precision', n_jobs=1)
print(f'CV PR-AUC per fold: {np.round(scores, 4)}')
print(f'Mean CV PR-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

if scores.mean() < 0.75:
    print('WARNING: Below 0.75 target. Debug before proceeding.')
else:
    print('CV target met. Proceed to Cell 5.')


In [ ]:
# Cell 5 — Evaluation on held-out test set
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred)

print('=== TEST SET EVALUATION ===')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  ROC-AUC:   {auc:.4f}')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\n{classification_report(y_test, y_pred, target_names=["benign","phishing"])}')

assert f1  >= 0.70, f'F1 too low: {f1:.4f}'
assert auc >= 0.75, f'AUC too low: {auc:.4f}'
print('Minimum performance thresholds passed.')



In [ ]:
# Cell 6 — SHAP feature importance
os.makedirs(EVAL_DIR, exist_ok=True)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary bar plot — mean |SHAP| per feature
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=TRAIN_FEATURES,
                  plot_type='bar',
                  show=False)
plt.tight_layout()
bar_path = f'{EVAL_DIR}/contract_shap_bar.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bar_path}')

# Beeswarm plot
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=TRAIN_FEATURES,
                  show=False)
plt.tight_layout()
bee_path = f'{EVAL_DIR}/contract_shap_beeswarm.png'
plt.savefig(bee_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bee_path}')

# Top features by mean |SHAP|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
print('\nFeature importance (mean |SHAP|):')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. {TRAIN_FEATURES[idx]:<45}  {mean_abs_shap[idx]:.5f}')

print('Cell 6 SHAP complete.')


In [ ]:
# Cell 7 — Save model and verify
MODEL_PATH = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
joblib.dump(model, MODEL_PATH)
print(f'Saved: {MODEL_PATH}')

loaded     = joblib.load(MODEL_PATH)
test_preds = loaded.predict_proba(X_test[:5])[:, 1]
print(f'Verify predictions: {np.round(test_preds, 4)}')
print('Model save verified.')
print('Notebook 05 complete.')


In [ ]:
# ── Live Contract Prediction ──────────────────────────────────────────────────
import requests, json, re, os, time, joblib
import numpy as np

CONTRACT_ADDRESS  = '0x0000626d6DC72989e3809920C67D01a7fe030000'   # <-- paste contract address here
ETHERSCAN_API_KEY = 'WI9RMPKI74VKDCZ8PHUYF1KK36RV8CVZN2'         # <-- paste your Etherscan key here
RUN_SLITHER       = False                 # True adds ~2 min for verified contracts
MODEL_PATH        = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'

# ── Load model artifact ───────────────────────────────────────────────────────
artifact         = joblib.load(MODEL_PATH)
pred_model       = artifact['model']
threshold        = artifact['threshold']
train_features   = artifact['train_features']

# ── Fetch contract data from Etherscan ────────────────────────────────────────
def fetch_contract(address, api_key):
    r = requests.get('https://api.etherscan.io/v2/api', params={
        'chainid': 1, 'module': 'contract',
        'action': 'getsourcecode',
        'address': address, 'apikey': api_key
    }, timeout=15)
    time.sleep(0.25)
    result = r.json().get('result', [{}])[0]
    return result

print(f'Fetching data for {CONTRACT_ADDRESS} ...')
data = fetch_contract(CONTRACT_ADDRESS, ETHERSCAN_API_KEY)

source_code  = data.get('SourceCode', '')
abi_str      = data.get('ABI', '[]')
compiler_ver = data.get('CompilerVersion', '')
is_verified  = int(bool(source_code and source_code not in ('', '0x')))

# ── Extract ABI features ──────────────────────────────────────────────────────
try:
    abi_list = json.loads(abi_str) if abi_str not in ('', 'Contract source code not verified') else []
except Exception:
    abi_list = []

functions  = [item for item in abi_list if item.get('type') == 'function']
func_names = [f.get('name', '').lower() for f in functions]

abi_function_count             = len(functions)
external_public_function_count = sum(
    1 for f in functions if f.get('stateMutability') not in ['view', 'pure'])
approval_related_function_flag = int(any(
    any(kw in n for kw in ['approve','setallowance','increaseallowance','decreaseallowance'])
    for n in func_names))
permit_related_function_flag   = int(any('permit' in n for n in func_names))
setApprovalForAll_flag         = int(any(n == 'setapprovalforall' for n in func_names))

# ── Slither features ──────────────────────────────────────────────────────────
slither_warning_count_total         = 0
slither_low_level_call_count        = 0
slither_access_control_issues_count = 0

if RUN_SLITHER and is_verified and source_code:
    print('Running Slither (this may take up to 2 minutes)...')
    import tempfile, shutil, subprocess

    def install_solc(ver_str):
        match = re.search(r'v?([\d]+\.[\d]+\.[\d]+)', ver_str)
        ver   = match.group(1) if match else '0.8.19'
        os.environ['PATH'] += ':/root/.local/bin:/usr/local/bin'
        os.system(f'solc-select install {ver} 2>/dev/null && solc-select use {ver} 2>/dev/null')

    def run_slither_inline(source_str, address, compiler_ver):
        install_solc(compiler_ver)
        temp_dir = tempfile.mkdtemp()
        src = source_str.strip()
        if src.startswith('{{'):
            try:
                parsed = json.loads(src[1:-1])
                entry  = None
                for fname, content in parsed.get('sources', {}).items():
                    fp = os.path.join(temp_dir, os.path.basename(fname))
                    with open(fp, 'w') as f: f.write(content.get('content', ''))
                    if entry is None: entry = fp
            except Exception:
                shutil.rmtree(temp_dir, ignore_errors=True)
                return 0, 0, 0
        else:
            entry = os.path.join(temp_dir, f'{address}.sol')
            with open(entry, 'w') as f: f.write(src)

        runner = tempfile.mktemp(suffix='.py')
        output = tempfile.mktemp(suffix='.json')
        script = f"""
import json
try:
    from slither.slither import Slither
    sl      = Slither({repr(entry)})
    results = sl.run_detectors()
    total   = sum(len(r.get('elements',[])) for r in results)
    low_lvl = sum(len(r.get('elements',[])) for r in results if 'low-level-calls' in r.get('check',''))
    access  = sum(len(r.get('elements',[])) for r in results if 'access-control' in r.get('check','') or 'unprotected' in r.get('check',''))
    with open({repr(output)},'w') as f: json.dump({{'t':total,'l':low_lvl,'a':access}},f)
except Exception:
    with open({repr(output)},'w') as f: json.dump({{'t':0,'l':0,'a':0}},f)
"""
        with open(runner, 'w') as f: f.write(script)
        try:
            subprocess.run(['python3', runner], timeout=120, capture_output=True)
            with open(output) as f: d = json.load(f)
            return d.get('t',0), d.get('l',0), d.get('a',0)
        except Exception:
            return 0, 0, 0
        finally:
            for fp in [runner, output]:
                try: os.unlink(fp)
                except: pass
            shutil.rmtree(temp_dir, ignore_errors=True)

    slither_warning_count_total, slither_low_level_call_count, \
        slither_access_control_issues_count = run_slither_inline(
            source_code, CONTRACT_ADDRESS.lower(), compiler_ver)
    print(f'Slither done — warnings={slither_warning_count_total}  '
          f'low-level={slither_low_level_call_count}  '
          f'access={slither_access_control_issues_count}')

# ── Build feature vector ──────────────────────────────────────────────────────
feature_map = {
    'is_verified':                           is_verified,
    'abi_function_count':                    abi_function_count,
    'external_public_function_count':        external_public_function_count,
    'approval_related_function_flag':        approval_related_function_flag,
    'permit_related_function_flag':          permit_related_function_flag,
    'setApprovalForAll_flag':                setApprovalForAll_flag,
    'slither_warning_count_total':           slither_warning_count_total,
    'slither_low_level_call_count':          slither_low_level_call_count,
    'slither_access_control_issues_count':   slither_access_control_issues_count
}

X_live = np.array([[feature_map[f] for f in train_features]])

# ── Predict ───────────────────────────────────────────────────────────────────
prob  = pred_model.predict_proba(X_live)[0, 1]
label = int(prob >= threshold)

print()
print('=' * 50)
print(f'Contract : {CONTRACT_ADDRESS}')
print(f'Verified : {"Yes" if is_verified else "No"}')
print(f'ABI fns  : {abi_function_count}  (external: {external_public_function_count})')
print(f'Approval fn flag : {bool(approval_related_function_flag)}')
print('-' * 50)
print(f'Phishing probability : {prob:.4f}')
print(f'Threshold            : {threshold:.4f}')
print(f'Prediction           : {"⚠ PHISHING" if label == 1 else "✓ BENIGN"}')
print('=' * 50)

if not RUN_SLITHER and is_verified:
    print('Note: Slither not run — set RUN_SLITHER=True for full analysis.')
